In [1]:
import pandas as pd
import numpy as np
import random

from faker import Faker
from datetime import datetime, timedelta

In [2]:
fake = Faker()

In [3]:
NUM_USERS = 100
LOGINS_PER_USER = 100

In [4]:
countries = [
    "India",
    "USA",
    "Germany",
    "Canada",
    "UK",
    "Australia"
]

In [5]:
devices = [
    "Windows",
    "Mac",
    "Linux",
    "Android",
    "iPhone"
]

In [6]:
browsers = [
    "Chrome",
    "Firefox",
    "Edge",
    "Safari"
]

In [7]:
user_profiles = {}

user_types = [

    "corporate",

    "developer",

    "mobile",

    "traveler",

    "high_risk"

]

for i in range(1, NUM_USERS + 1):

    user_id = f"user_{i}"

    user_type = random.choice(user_types)

    if user_type == "corporate":

        profile = {

            "type": "corporate",

            "country": random.choice([
                "India",
                "USA",
                "UK"
            ]),

            "device": "Windows",

            "browser": "Chrome",

            "start_hour": 8,

            "end_hour": 18,

            "avg_failed_attempts": 1
        }

    elif user_type == "developer":

        profile = {

            "type": "developer",

            "country": random.choice([
                "Germany",
                "Canada",
                "India"
            ]),

            "device": "Linux",

            "browser": "Firefox",

            "start_hour": 10,

            "end_hour": 2,

            "avg_failed_attempts": 2
        }

    elif user_type == "mobile":

        profile = {

            "type": "mobile",

            "country": random.choice(countries),

            "device": random.choice([
                "Android",
                "iPhone"
            ]),

            "browser": "Safari",

            "start_hour": 6,

            "end_hour": 23,

            "avg_failed_attempts": 1
        }

    elif user_type == "traveler":

        profile = {

            "type": "traveler",

            "country": random.choice(countries),

            "device": random.choice(devices),

            "browser": random.choice(browsers),

            "start_hour": 5,

            "end_hour": 23,

            "avg_failed_attempts": 2
        }

    else:

        profile = {

            "type": "high_risk",

            "country": random.choice(countries),

            "device": random.choice(devices),

            "browser": random.choice(browsers),

            "start_hour": 0,

            "end_hour": 23,

            "avg_failed_attempts": 4
        }

    user_profiles[user_id] = profile

In [8]:
list(user_profiles.items())[:5]

[('user_1',
  {'type': 'mobile',
   'country': 'UK',
   'device': 'iPhone',
   'browser': 'Safari',
   'start_hour': 6,
   'end_hour': 23,
   'avg_failed_attempts': 1}),
 ('user_2',
  {'type': 'traveler',
   'country': 'India',
   'device': 'Android',
   'browser': 'Firefox',
   'start_hour': 5,
   'end_hour': 23,
   'avg_failed_attempts': 2}),
 ('user_3',
  {'type': 'traveler',
   'country': 'USA',
   'device': 'iPhone',
   'browser': 'Edge',
   'start_hour': 5,
   'end_hour': 23,
   'avg_failed_attempts': 2}),
 ('user_4',
  {'type': 'mobile',
   'country': 'Canada',
   'device': 'iPhone',
   'browser': 'Safari',
   'start_hour': 6,
   'end_hour': 23,
   'avg_failed_attempts': 1}),
 ('user_5',
  {'type': 'traveler',
   'country': 'Australia',
   'device': 'Mac',
   'browser': 'Chrome',
   'start_hour': 5,
   'end_hour': 23,
   'avg_failed_attempts': 2})]

In [9]:
data = []

for user_id, profile in user_profiles.items():

    for _ in range(LOGINS_PER_USER):

        if profile["start_hour"] <= profile["end_hour"]:

            login_hour = random.randint(
                profile["start_hour"],
                profile["end_hour"]
            )

        else:

            possible_hours = list(
                range(profile["start_hour"], 24)
            ) + list(
                range(0, profile["end_hour"] + 1)
            )

            login_hour = random.choice(possible_hours)

        timestamp = fake.date_time_between(
            start_date='-30d',
            end_date='now'
        ).replace(hour=login_hour)

        failed_attempts = max(

            0,

            int(
                np.random.normal(
                    profile["avg_failed_attempts"],
                    1
                )
            )

        )

        login_status = "Success"

        data.append([
            user_id,
            timestamp,
            profile["country"],
            profile["device"],
            profile["browser"],
            failed_attempts,
            login_status,
            0
        ])

In [10]:
columns = [
    "user_id",
    "timestamp",
    "country",
    "device_type",
    "browser",
    "failed_attempts",
    "login_status",
    "is_anomaly"
]

df = pd.DataFrame(data, columns=columns)

In [11]:
df.head()

,user_id,timestamp,country,device_type,browser,failed_attempts,login_status,is_anomaly
0,user_1,2026-05-11 21:30:09,UK,iPhone,Safari,1,Success,0
1,user_1,2026-05-20 13:28:55,UK,iPhone,Safari,0,Success,0
2,user_1,2026-05-04 17:39:55,UK,iPhone,Safari,3,Success,0
3,user_1,2026-05-06 10:59:42,UK,iPhone,Safari,0,Success,0
4,user_1,2026-05-16 13:34:28,UK,iPhone,Safari,1,Success,0


In [12]:
anomaly_indices = random.sample(
    range(len(df)),
    500
)

In [13]:
for idx in anomaly_indices:

    current_user = df.loc[idx, "user_id"]

    normal_profile = user_profiles[current_user]

    suspicious_country = random.choice([
        c for c in countries
        if c != normal_profile["country"]
    ])

    suspicious_device = random.choice([
        d for d in devices
        if d != normal_profile["device"]
    ])

    suspicious_browser = random.choice([
        b for b in browsers
        if b != normal_profile["browser"]
    ])

    suspicious_hour = random.choice([
        random.randint(0, 4),
        random.randint(23, 23)
    ])

    suspicious_timestamp = fake.date_time_between(
        start_date='-30d',
        end_date='now'
    ).replace(hour=suspicious_hour)

    df.loc[idx, "country"] = suspicious_country

    df.loc[idx, "device_type"] = suspicious_device

    df.loc[idx, "browser"] = suspicious_browser

    df.loc[idx, "timestamp"] = suspicious_timestamp

    df.loc[idx, "failed_attempts"] = random.randint(8, 20)

    df.loc[idx, "login_status"] = "Failed"

    df.loc[idx, "is_anomaly"] = 1

In [14]:
df.head()

,user_id,timestamp,country,device_type,browser,failed_attempts,login_status,is_anomaly
0,user_1,2026-04-28 02:09:32,Australia,Mac,Edge,13,Failed,1
1,user_1,2026-05-20 13:28:55,UK,iPhone,Safari,0,Success,0
2,user_1,2026-05-04 17:39:55,UK,iPhone,Safari,3,Success,0
3,user_1,2026-05-06 10:59:42,UK,iPhone,Safari,0,Success,0
4,user_1,2026-05-16 13:34:28,UK,iPhone,Safari,1,Success,0


In [15]:
df["failed_attempts"].describe()

count    10000.000000
mean         2.002800
std          3.149569
min          0.000000
25%          0.000000
50%          1.000000
75%          2.000000
max         20.000000
Name: failed_attempts, dtype: float64

In [16]:
df.groupby("user_id")["failed_attempts"].mean().head(10)

user_id
user_1      1.83
user_10     1.22
user_100    2.19
user_11     1.00
user_12     1.47
user_13     1.25
user_14     1.74
user_15     3.83
user_16     1.37
user_17     4.08
Name: failed_attempts, dtype: float64

In [17]:
df.to_csv(
    "login_behavior_dataset.csv",
    index=False
)

print("Behavior-Based Dataset Saved!")

Behavior-Based Dataset Saved!
